In [ ]:
import json
import itertools
import math
from pathlib import Path

import matlab.engine


# =========================================
# 1. 사용자 설정
# =========================================
MODEL_FILE = "Matlab/P1.slx"
MODEL_NAME = "P1"
SAVE_PATH = Path("msd_ccd_results.json")
STOP_TIME = 30

# CCD 대상 요인: 3개
FACTOR_SPACE = {
    "Kp": (5.0, 20.0),
    "Ki": (1.0, 15.0),
    "Kd": (0.1, 5.0),
}

# 고정 파라미터
FIXED_PARAMS = {
    "m": 1.0,
    "c": 2.0,
    "k": 5.0,
}

SAVE_SIGNALS = False

# center point 반복 수
N_CENTER = 5

# rotatable CCD용 alpha
N_FACTORS = len(FACTOR_SPACE)
ALPHA = (2 ** N_FACTORS) ** 0.25  # k=3이면 약 1.68179


# =========================================
# 2. 변환 유틸
# =========================================
def matlab_scalar_to_float(x):
    if isinstance(x, (int, float)):
        return float(x)

    try:
        data = list(x)
        if len(data) == 1:
            if isinstance(data[0], list) and len(data[0]) == 1:
                return float(data[0][0])
            if not isinstance(data[0], list):
                return float(data[0])
    except Exception:
        pass

    return float(x)


def matlab_vector_to_1d_list(x, ndigits=8):
    if isinstance(x, (int, float)):
        return [round(float(x), ndigits)]

    data = list(x)

    if len(data) == 0:
        return []

    if isinstance(data[0], list) and len(data[0]) == 1:
        return [round(float(row[0]), ndigits) for row in data]

    if isinstance(data[0], list) and len(data) == 1:
        return [round(float(v), ndigits) for v in data[0]]

    if not isinstance(data[0], list):
        return [round(float(v), ndigits) for v in data]

    return [[round(float(v), ndigits) for v in row] for row in data]


# =========================================
# 3. CCD 생성
# =========================================
def build_ccd_coded_points(factor_names, alpha=1.68179, n_center=5):
    k = len(factor_names)

    # 1) factorial points: (-1, +1)^k
    factorial_points = list(itertools.product([-1.0, 1.0], repeat=k))

    # 2) axial points: (±alpha, 0, 0, ...)
    axial_points = []
    for i in range(k):
        pt_plus = [0.0] * k
        pt_minus = [0.0] * k
        pt_plus[i] = alpha
        pt_minus[i] = -alpha
        axial_points.append(tuple(pt_plus))
        axial_points.append(tuple(pt_minus))

    # 3) center points
    center_points = [tuple([0.0] * k) for _ in range(n_center)]

    coded_points = factorial_points + axial_points + center_points
    return coded_points


def coded_to_real_params(coded_point, factor_space):
    factor_names = list(factor_space.keys())
    params = {}

    for i, name in enumerate(factor_names):
        low, high = factor_space[name]
        center = (low + high) / 2.0
        half_range = (high - low) / 2.0
        params[name] = center + coded_point[i] * half_range

    return params


def generate_ccd_design(factor_space, fixed_params, alpha=1.68179, n_center=5):
    factor_names = list(factor_space.keys())
    coded_points = build_ccd_coded_points(factor_names, alpha=alpha, n_center=n_center)

    design = []
    for idx, coded in enumerate(coded_points):
        real_params = coded_to_real_params(coded, factor_space)
        real_params.update(fixed_params)

        design.append({
            "sample_id": idx,
            "coded_factors": {factor_names[i]: coded[i] for i in range(len(factor_names))},
            "params": real_params,
        })

    return design


# =========================================
# 4. 시뮬레이션 유틸
# =========================================
def set_workspace_params(eng, params: dict):
    for key, value in params.items():
        eng.workspace[key] = float(value)


def run_simulation(eng, model_name: str, stop_time=30):
    eng.set_param(model_name, "StopTime", str(stop_time), nargout=0)
    eng.eval(f"out = sim('{model_name}');", nargout=0)


def compute_stepinfo_position(eng):
    eng.eval("info_pos = stepinfo(out.pos_out(:), out.t_out(:));", nargout=0)

    fields = [
        "RiseTime", "TransientTime", "SettlingTime",
        "SettlingMin", "SettlingMax", "Overshoot",
        "Undershoot", "Peak", "PeakTime"
    ]

    return {
        field: matlab_scalar_to_float(eng.eval(f"info_pos.{field}", nargout=1))
        for field in fields
    }


def collect_signals(eng):
    return {
        "t_out": matlab_vector_to_1d_list(eng.eval("out.t_out", nargout=1)),
        "pos_out": matlab_vector_to_1d_list(eng.eval("out.pos_out", nargout=1)),
        "vel_out": matlab_vector_to_1d_list(eng.eval("out.vel_out", nargout=1)),
    }


def run_one_case(eng, model_name: str, case_info: dict, save_signals: bool = False):
    params = case_info["params"]

    set_workspace_params(eng, params)
    run_simulation(eng, model_name, stop_time=STOP_TIME)

    result = {
        "sample_id": case_info["sample_id"],
        "coded_factors": case_info["coded_factors"],
        "params": {k: float(v) for k, v in params.items()},
        "position_metrics": compute_stepinfo_position(eng),
    }

    if save_signals:
        result["signals"] = collect_signals(eng)

    return result


# =========================================
# 5. 메인 실행
# =========================================
def main():
    eng = matlab.engine.start_matlab()

    try:
        eng.load_system(MODEL_FILE, nargout=0)

        stepinfo_path = eng.eval("which('stepinfo','-all')", nargout=1)
        has_control_toolbox = eng.eval("license('test','Control_Toolbox')", nargout=1)

        print("stepinfo path:")
        print(stepinfo_path)
        print("Control Toolbox available:", has_control_toolbox)

        if not has_control_toolbox:
            raise RuntimeError("Control System Toolbox 라이선스를 사용할 수 없습니다.")

        design = generate_ccd_design(
            factor_space=FACTOR_SPACE,
            fixed_params=FIXED_PARAMS,
            alpha=ALPHA,
            n_center=N_CENTER,
        )

        print(f"Total CCD runs: {len(design)}")
        results = []

        for i, case_info in enumerate(design):
            try:
                result = run_one_case(
                    eng=eng,
                    model_name=MODEL_NAME,
                    case_info=case_info,
                    save_signals=SAVE_SIGNALS,
                )
                results.append(result)

                print(f"[{i+1}/{len(design)}] success")
                print("  coded:", result["coded_factors"])
                print("  params:", result["params"])
                print("  metrics:", result["position_metrics"])

            except Exception as e:
                print(f"[{i+1}/{len(design)}] failed: {e}")

        with open(SAVE_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        print(f"\nSaved JSON: {SAVE_PATH.resolve()}")
        print(f"Total successful cases: {len(results)}")

    finally:
        try:
            eng.quit()
        except Exception:
            pass


if __name__ == "__main__":
    main()

stepinfo path:
['C:\\Program Files\\MATLAB\\R2024a\\toolbox\\shared\\controllib\\engine\\stepinfo.m', 'C:\\Program Files\\MATLAB\\R2024a\\toolbox\\control\\ctrlanalysis\\@DynamicSystem\\stepinfo.m']
Control Toolbox available: 1.0
Total CCD runs: 19
[1/19] success
  coded: {'Kp': -1.0, 'Ki': -1.0, 'Kd': -1.0}
  params: {'Kp': 5.0, 'Ki': 1.0, 'Kd': 0.09999999999999964, 'm': 1.0, 'c': 2.0, 'k': 5.0}
  metrics: {'RiseTime': 13.718256751697393, 'TransientTime': 24.056119972045014, 'SettlingTime': 24.056119972045014, 'SettlingMin': 0.8845459067317525, 'SettlingMax': 0.9766104979754663, 'Overshoot': 0.0, 'Undershoot': -0.0, 'Peak': 0.9766104979754663, 'PeakTime': 30.0}
[2/19] success
  coded: {'Kp': -1.0, 'Ki': -1.0, 'Kd': 1.0}
  params: {'Kp': 5.0, 'Ki': 1.0, 'Kd': 5.0, 'm': 1.0, 'c': 2.0, 'k': 5.0}
  metrics: {'RiseTime': 14.28808321253707, 'TransientTime': 24.115194849583716, 'SettlingTime': 24.115194849583716, 'SettlingMin': 0.8803937730252466, 'SettlingMax': 0.9780047898838787, 'Overshoo